# Chapter 9 — Multimodal Large Language Models
## Practice Notebook

**Book:** Hands-On Large Language Models — Jay Alammar & Maarten Grootendorst (O'Reilly)  
**Chapter:** 9 — Multimodal Large Language Models  
**Companion notes:** `notes/ch09-multimodal-llms.md`

---

This notebook builds your multimodal skills in five stages:

- **Part 0 — Theory Warm-Up** (no GPU needed): implement core concepts from scratch with numpy
- **Part 1 — Vision Transformer Concepts**: manually patch an image and apply a linear projection
- **Part 2 — OpenCLIP**: load CLIP and compute a cross-modal similarity matrix
- **Part 3 — CLIP via sentence-transformers**: simpler API, same result
- **Part 4 — BLIP-2: Preprocessing**: load the model and understand the image pipeline
- **Part 5 — BLIP-2: Use Cases**: image captioning, visual QA, and multimodal chat

> **Before you start:** read `notes/ch09-multimodal-llms.md` sections 1–4 before Part 0–3,  
> and sections 5–9 before Parts 4–5.

In [ ]:
# Optional: uncomment to install dependencies in Colab
# !pip install transformers sentence-transformers open_clip_torch torch torchvision pillow requests matplotlib

---
## Part 0 — Theory Warm-Up

These exercises use only `numpy`. No model downloads, no GPU required.  
Complete these before running any model cells — they build the mathematical intuition you need.

---
### T1 — Image Patch Tokenization from Scratch

The Vision Transformer's first step is cutting an image into non-overlapping square patches.  
Before using ViT, implement this yourself.

**Concept recap** (from notes section 2b):  
A 28×28 greyscale image with patch size 4×4 produces (28÷4) × (28÷4) = 7 × 7 = **49 patches**.  
Each patch is 4×4 = 16 raw pixel values. After flattening: shape `(49, 16)`.

**Task:** Write a function `image_to_patches(image, patch_size)` that:
1. Takes a 2D numpy array `image` of shape `(H, W)` and an integer `patch_size`
2. Divides the image into non-overlapping square patches
3. Flattens each patch into a 1D vector
4. Returns an array of shape `(num_patches, patch_size²)`
5. Prints the shape at every step

**Hint:** Use nested loops over patch grid positions, or `numpy` reshape + transpose.  
Expected output for a (28, 28) image with patch_size=4:
```
Image shape: (28, 28)
Patch grid: 7 × 7 = 49 patches
Each patch flattened: 16 values
Output shape: (49, 16)
```

In [ ]:
import numpy as np

def image_to_patches(image, patch_size):
    """
    Split a 2D greyscale image into non-overlapping square patches and flatten each.

    Args:
        image:      numpy array of shape (H, W)
        patch_size: integer side length of each square patch

    Returns:
        patches: numpy array of shape (num_patches, patch_size * patch_size)
    """
    # YOUR CODE HERE
    pass


# Test with a 28×28 random image and patch_size=4
np.random.seed(42)
fake_image = np.random.randint(0, 256, size=(28, 28), dtype=np.uint8)

patches = image_to_patches(fake_image, patch_size=4)

# Verify
assert patches.shape == (49, 16), f"Expected (49, 16), got {patches.shape}"
print("T1 passed — patches shape:", patches.shape)

---
### T2 — Contrastive Similarity Matrix from Scratch

CLIP's training objective requires computing cosine similarities between every image embedding and every text embedding in a batch — forming an N×N matrix where the diagonal should be highest.

**Concept recap** (from notes section 4d–4e):  
For L2-normalised vectors, cosine similarity = dot product.  
Formula: $\text{sim}(u, v) = \frac{u \cdot v}{\|u\| \cdot \|v\|}$

**Task:** Write a function `cosine_similarity_matrix(image_embs, text_embs)` that:
1. L2-normalises both sets of embeddings (divide each row by its norm)
2. Computes the full N×N cosine similarity matrix using only numpy
3. Prints the matrix with labels
4. Verifies that the diagonal values are the highest in each row

**Hint:**  
- Normalise with `v / np.linalg.norm(v, axis=-1, keepdims=True)`  
- The full matrix is `normalised_image_embs @ normalised_text_embs.T`  
- Expected: diagonal values ≈ highest in each row

In [ ]:
import numpy as np

def cosine_similarity_matrix(image_embs, text_embs):
    """
    Compute an N×N cosine similarity matrix between image and text embeddings.

    Args:
        image_embs: numpy array of shape (N, D) — one image embedding per row
        text_embs:  numpy array of shape (N, D) — one text embedding per row

    Returns:
        sim_matrix: numpy array of shape (N, N) where sim_matrix[i, j] = sim(text_i, image_j)
    """
    # YOUR CODE HERE
    pass


# Test with 3 paired embeddings (2D for easy manual verification)
# These match the dry-run in notes section 4e
image_embs = np.array([
    [0.90,  0.44],   # puppy image
    [0.10,  0.99],   # cat image
    [0.71, -0.71],   # car image
], dtype=np.float32)

text_embs = np.array([
    [0.85,  0.53],   # puppy caption
    [0.05,  1.00],   # cat caption
    [0.71, -0.70],   # car caption
], dtype=np.float32)

sim = cosine_similarity_matrix(image_embs, text_embs)

print("Similarity matrix (rows=text, cols=image):")
labels = ["puppy", "cat", "car"]
print(f"{'':12}" + "".join(f"{l:>10}" for l in labels))
for i, row_label in enumerate(labels):
    print(f"{row_label:12}" + "".join(f"{sim[i,j]:>10.3f}" for j in range(3)))

# Verify: diagonal should be max in each row
for i in range(3):
    assert sim[i, i] == sim[i].max(), f"Row {i}: diagonal is not max!"
print("T2 passed — diagonal is highest in every row")

---
## Part 1 — Vision Transformer Concepts

Before loading a full ViT, implement the two key steps manually: patch extraction and linear projection.

---
### 1.1 — Load an Image and Split It into Patches

Use your `image_to_patches` function from T1 on an actual image.  
Visualise the result as a grid of patches.

**Task:**
1. Load a small image (use the URL below or any PIL-loadable image)
2. Convert to greyscale and resize to 28×28
3. Apply `image_to_patches` with `patch_size=7` to get 4×4=16 patches
4. Visualise the 16 patches in a 4×4 matplotlib grid
5. Print the shapes at every step

**Hint:** Use `PIL.Image`, `numpy`, and `matplotlib.pyplot`.

```python
IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"
```

In [ ]:
from PIL import Image
from urllib.request import urlopen
import numpy as np
import matplotlib.pyplot as plt

IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"

# YOUR CODE HERE
# Step 1: Load image from URL using urlopen + PIL
# Step 2: Convert to greyscale and resize to 28×28
# Step 3: Convert to numpy array, print shape
# Step 4: Apply image_to_patches(image_array, patch_size=7)
#         Expected shape: (16, 49)  — 16 patches, each 7×7=49 pixels
# Step 5: Reshape each patch back to (7, 7) for display
# Step 6: Plot patches in a 4×4 grid using plt.subplot

---
### 1.2 — Linear Projection of Patch Embeddings

In ViT, each flattened patch is multiplied by a learned weight matrix $W_p$ to produce a $d_{model}$-dimensional embedding.  
The projection is: $e_i = \text{flatten}(P_i) \cdot W_p$

**Task:**
1. Using the 16 patches from Part 1.1 (shape `(16, 49)`), apply a `torch.nn.Linear(49, 8)` layer
2. Print the shape before and after projection
3. Prepend a randomly initialised `[CLASS]` token (shape `(1, 8)`) to get shape `(17, 8)`
4. Print the final sequence shape and explain what each number means

**Hint:**  
- Convert numpy patches to a `torch.Tensor` with `torch.from_numpy(...).float()`  
- `nn.Linear(in_features=49, out_features=8)` projects each patch from 49-dim to 8-dim  
- CLS token: `cls_token = torch.zeros(1, 8)` (initialised to zero at start of training)  
- Stack: `torch.cat([cls_token, patch_embeddings], dim=0)` — shape `(17, 8)`

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# YOUR CODE HERE
# Step 1: Convert patches (numpy, shape (16, 49)) to torch tensor
# Step 2: Define nn.Linear(49, 8) as the projection layer
# Step 3: Apply projection — print input and output shapes
#         Input:  (16, 49)   — 16 patches, each 49 raw pixel values
#         Output: (16, 8)    — 16 patch embeddings, each 8-dimensional
# Step 4: Create CLS token of shape (1, 8)
# Step 5: Prepend CLS token — final shape should be (17, 8)
# Step 6: Print and explain: "17 tokens (1 CLS + 16 patches), each 8-dim"

---
## Part 2 — OpenCLIP

Now use a real pretrained CLIP model to compute cross-modal embeddings.  
We'll work with 3 image-caption pairs and build a 3×3 similarity matrix — matching the book's Figure 9-14.

---
### 2.1 — Load the CLIP Model

CLIP requires three components: a tokenizer (for text), a processor (for images), and the main model.

**Task:** Load `openai/clip-vit-base-patch32` from HuggingFace and print each component's type.

In [ ]:
from transformers import CLIPTokenizerFast, CLIPProcessor, CLIPModel

MODEL_ID = "openai/clip-vit-base-patch32"

# YOUR CODE HERE
# Load: CLIPTokenizerFast, CLIPProcessor, CLIPModel — all from MODEL_ID
# Print the type of each component so you can see what you loaded

# clip_tokenizer = ...
# clip_processor = ...
# clip_model = ...

---
### 2.2 — Generate Text Embeddings

Use the tokenizer and model to produce a 512-dimensional embedding for each of 3 captions.

**Task:**
1. Tokenize all 3 captions in a single batch call using `clip_tokenizer`
2. Pass through `clip_model.get_text_features()` to get text embeddings
3. Print the shape — expected: `torch.Size([3, 512])`
4. L2-normalise the embeddings

**Hint:** `clip_tokenizer(captions, padding=True, return_tensors="pt")` handles batched input.

In [ ]:
import torch

# The same 3 captions used in the book (Figure 9-14)
captions = [
    "A puppy playing in the snow",
    "A pixelated image of a cute cat",
    "A supercar on the road with sunset in background",
]

# YOUR CODE HERE
# Step 1: Tokenize all 3 captions in one batch
# Step 2: Extract text embeddings — shape should be (3, 512)
# Step 3: L2-normalise: divide each row by its norm
#         Hint: text_embs /= text_embs.norm(dim=-1, keepdim=True)
# Step 4: Print shape and the first 5 values of caption 0

---
### 2.3 — Generate Image Embeddings

Load 3 images (puppy, cat, car), preprocess each, and produce image embeddings.

**Task:**
1. Load each image from the URLs below using `PIL.Image` + `urllib.request.urlopen`
2. Preprocess all 3 images in one batch using `clip_processor`
3. Extract image embeddings using `clip_model.get_image_features()`
4. L2-normalise the embeddings
5. Print the shape — expected: `torch.Size([3, 512])`

In [ ]:
from PIL import Image
from urllib.request import urlopen

# Image URLs (same subject matter as book Figure 9-12 to 9-14)
image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2009_Audi_TT_--_10-30-2011.jpg/320px-2009_Audi_TT_--_10-30-2011.jpg",
]
image_labels = ["puppy", "cat", "car"]

# YOUR CODE HERE
# Step 1: Load each URL with urlopen and convert to RGB PIL Image
# Step 2: Preprocess — clip_processor(text=None, images=images, return_tensors="pt")["pixel_values"]
#         Expected pixel_values shape: (3, 3, 224, 224)
# Step 3: Extract image embeddings — shape: (3, 512)
# Step 4: L2-normalise
# Step 5: Print shape

---
### 2.4 — Compute and Display the 3×3 Similarity Matrix

**Task:**
1. Compute the 3×3 similarity matrix: `sim_matrix = text_embs @ image_embs.T`
2. Display it as a matplotlib heatmap (matching book Figure 9-14)
3. Print which (text, image) pair has the highest similarity in each row
4. Verify that the diagonal values are highest in each row

**Expected values** (from book Figure 9-14):
```
              puppy  cat   car
puppy text  [ 0.33  0.19  0.11 ]
cat text    [ 0.15  0.35  0.09 ]
car text    [ 0.08  0.13  0.31 ]
```

**Hint:** Use `plt.imshow`, `plt.xticks`, `plt.yticks`, and annotate each cell with the score.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# YOUR CODE HERE
# Step 1: Compute similarity matrix (both embs should already be normalised)
#         sim_matrix = text_embs @ image_embs.T   — shape: (3, 3)
# Step 2: Convert to numpy for display
# Step 3: Plot as heatmap with annotated values
#         - x-axis: image labels
#         - y-axis: caption labels
#         - annotate each cell: f"{score:.2f}"
# Step 4: Print the highest-scoring pair per row

---
## Part 3 — CLIP via sentence-transformers

The `sentence-transformers` library provides a simpler API for CLIP that handles normalisation automatically.  
You should get the **same similarity scores** as Part 2 — this is a good sanity check.

---
### 3.1 — Load the CLIP Model via SentenceTransformer

In [ ]:
from sentence_transformers import SentenceTransformer

# YOUR CODE HERE
# Load: SentenceTransformer("clip-ViT-B-32")
# Print the model type

# st_model = ...

---
### 3.2 — Encode Images and Captions

**Task:**
1. Encode the 3 PIL images (reuse from Part 2) using `st_model.encode(images)`
2. Encode the 3 captions using `st_model.encode(captions)`
3. Print both shapes — expected: `(3, 512)` each

**Hint:** `sentence-transformers` accepts PIL Image objects directly for image encoding. No separate preprocessing step needed.

In [ ]:
# YOUR CODE HERE
# Step 1: st_image_embs = st_model.encode(images)   — list of PIL images from Part 2
# Step 2: st_text_embs  = st_model.encode(captions) — list of strings
# Step 3: Print both shapes

---
### 3.3 — Compare Results to Part 2

**Task:**
1. Compute the similarity matrix using `util.cos_sim(st_text_embs, st_image_embs)`
2. Print the matrix
3. Compare to the Part 2 matrix — values should be very close (small floating point differences)

**Why they might differ slightly:** The `sentence-transformers` API normalises internally. Small numerical differences from float32 precision are expected. The ranking of diagonal vs. off-diagonal should be identical.

In [ ]:
from sentence_transformers import util

# YOUR CODE HERE
# Step 1: Compute sim_matrix_st = util.cos_sim(st_text_embs, st_image_embs)
# Step 2: Print the matrix with row/col labels
# Step 3: Print the maximum absolute difference vs. Part 2 matrix
#         Should be < 0.01 for all entries

---
## Part 4 — BLIP-2: Preprocessing

> **Note:** Parts 4 and 5 require a GPU with ~16GB VRAM (or Google Colab A100/T4).  
> The model is `Salesforce/blip2-opt-2.7b` — 2.7B parameters for the LLM alone.

---
### 4.1 — Load the BLIP-2 Processor and Model

**Task:**
1. Load `AutoProcessor` from `"Salesforce/blip2-opt-2.7b"`
2. Load `Blip2ForConditionalGeneration` in `torch.float16` precision
3. Move the model to GPU
4. Print the three key components: `model.vision_model`, `model.language_model`, and the Q-Former

**Hint:** To access the Q-Former: `model.qformer`

In [ ]:
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# YOUR CODE HERE
# Step 1: blip_processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
# Step 2: model = Blip2ForConditionalGeneration.from_pretrained(..., torch_dtype=torch.float16)
# Step 3: model.to(device)
# Step 4: Print type(model.vision_model), type(model.language_model), type(model.qformer)

---
### 4.2 — Preprocess an Image

Run an image through the BLIP-2 processor and inspect what shape the model receives.

**Task:**
1. Load the supercar image (see URL below)
2. Print the original image size (PIL `.size` attribute gives `(width, height)`)
3. Run `blip_processor(image, return_tensors="pt")` to get `pixel_values`
4. Print the shape of `pixel_values` — expected: `torch.Size([1, 3, 224, 224])`
5. Explain in a comment why it is always `224×224` regardless of input size

In [ ]:
from PIL import Image
from urllib.request import urlopen

CAR_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2009_Audi_TT_--_10-30-2011.jpg/640px-2009_Audi_TT_--_10-30-2011.jpg"

# YOUR CODE HERE
# Step 1: Load image from CAR_URL, convert to RGB
# Step 2: Print original size — e.g. (640, 427)
# Step 3: Process with blip_processor
# Step 4: Print pixel_values.shape
# Step 5: Add a comment: "224×224 because the frozen ViT was pre-trained at this resolution"

---
### 4.3 — Visualise the Preprocessed Image

The BLIP-2 processor normalises pixel values using ImageNet mean and std.  
Visualising the preprocessed tensor (before the ViT sees it) gives intuition about what the model actually receives.

**Task:**
1. Extract `pixel_values` tensor from the processed inputs, shape `[1, 3, 224, 224]`
2. Remove the batch dimension: `pixel_values[0]` gives `[3, 224, 224]`
3. Permute to `[224, 224, 3]` for matplotlib
4. Plot both the original image and the preprocessed image side-by-side

**Note:** The preprocessed values are normalised (mean ≈ 0), so the image will look faded/blue-shifted — this is expected.

In [ ]:
import matplotlib.pyplot as plt

# YOUR CODE HERE
# Step 1: Extract pixel_values from inputs dict, move to CPU
# Step 2: Remove batch dim, permute channels last: shape (224, 224, 3)
# Step 3: Clamp values to [0, 1] for display (normalised values may go outside this range)
#         Hint: .clamp(0, 1) or use a manual rescale
# Step 4: fig, (ax1, ax2) = plt.subplots(1, 2) — left: original, right: preprocessed

In [ ]:
# VRAM cleanup between Parts 4 and 5
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("VRAM cleared")

---
## Part 5 — BLIP-2: Use Cases

---
### 5.1 — Image Captioning

Generate a text caption from the image alone — no text prompt.

**Task:**
1. Process the car image (no text argument)
2. Call `model.generate(**inputs, max_new_tokens=20)`
3. Decode the generated token IDs using `blip_processor.batch_decode(..., skip_special_tokens=True)`
4. Print the caption
5. Try one more image of your choice and print its caption

**Expected output:** Something like `"an orange supercar driving on the road at sunset"`

In [ ]:
# YOUR CODE HERE
# Step 1: inputs = blip_processor(image, return_tensors="pt").to(device, torch.float16)
# Step 2: generated_ids = model.generate(**inputs, max_new_tokens=20)
# Step 3: generated_text = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)
# Step 4: Print the caption string

---
### 5.2 — Visual Question Answering

Ask a specific question about the image. The key change is adding a `text` argument to the processor.

**Task:**
1. Using the car image, ask at least **two different questions**:
   - `"Question: What color is the car? Answer:"`
   - `"Question: What time of day does this appear to be? Answer:"`
2. For each question, process `(image, text=prompt)` together and generate the answer
3. Print each question and its answer

**Hint:** The prompt format `"Question: {q} Answer:"` is the expected template for OPT-based BLIP-2.

In [ ]:
# YOUR CODE HERE
# For each question:
# Step 1: prompt = "Question: {your question} Answer:"
# Step 2: inputs = blip_processor(image, text=prompt, return_tensors="pt").to(device, torch.float16)
# Step 3: generated_ids = model.generate(**inputs, max_new_tokens=20)
# Step 4: Decode and print the answer

---
### 5.3 — Chat-Style Multi-Turn Prompting

BLIP-2 maintains conversational context by accumulating the full exchange history in the prompt string.  
The image is fixed — the same 32 soft visual prompts anchor every turn.

**Task:** Implement a 3-turn conversation about the car image:
1. Turn 1: `"Question: Write down what you see. Answer:"`
2. Turn 2: Build a new prompt that includes Turn 1's Q+A, then adds a follow-up question
3. Turn 3: Build a new prompt that includes Turns 1 and 2's Q+A, then adds a third question
4. Print all three exchanges

**Hint:** After each turn, append to the prompt string:  
```python
prompt += f" {answer}. Question: {next_question} Answer:"
```

In [ ]:
def blip2_generate(model, processor, image, prompt, device, max_new_tokens=30):
    """Helper: run one generation step and return the answer string."""
    # YOUR CODE HERE
    # Step 1: Process image + text prompt
    # Step 2: Generate
    # Step 3: Decode and strip whitespace
    # Step 4: Return the answer string
    pass


# Run a 3-turn conversation
# YOUR CODE HERE
# Turn 1: initial question
# Turn 2: follow-up (include Turn 1 in prompt)
# Turn 3: deeper follow-up (include Turns 1+2 in prompt)
# Print each USER: / BLIP-2: exchange

---
### 5.4 — Theory Exercise: Conversation Prompt Formatter

No model needed — this is a pure Python string formatting exercise.

**Background:** In BLIP-2 multi-turn chat, each new turn requires constructing a prompt string that includes all previous exchanges.  
Currently we're doing this manually with string concatenation.  
A cleaner approach is a function that takes the conversation history and formats it automatically.

**Task:** Write a function `format_blip2_prompt(history, new_question)` where:
- `history` is a list of `(question, answer)` tuples (may be empty for the first turn)
- `new_question` is the next question string
- The function returns the full BLIP-2 prompt string

**Expected behaviour:**
```python
format_blip2_prompt([], "What is this?")
# → "Question: What is this? Answer:"

format_blip2_prompt(
    [("What is this?", "A red car")],
    "What color is it?"
)
# → "Question: What is this? Answer: A red car. Question: What color is it? Answer:"

format_blip2_prompt(
    [("What is this?", "A red car"), ("What color is it?", "Red")],
    "How fast can it go?"
)
# → "Question: What is this? Answer: A red car. Question: What color is it? Answer: Red. Question: How fast can it go? Answer:"
```

In [ ]:
def format_blip2_prompt(history, new_question):
    """
    Build a BLIP-2 multi-turn conversation prompt from history.

    Args:
        history:      list of (question_str, answer_str) tuples
        new_question: string — the next question to ask

    Returns:
        prompt: string in BLIP-2 Q/A format, ending with "Answer:"
    """
    # YOUR CODE HERE
    pass


# Test cases
result_1 = format_blip2_prompt([], "What is this?")
assert result_1 == "Question: What is this? Answer:", f"Got: {result_1!r}"

result_2 = format_blip2_prompt(
    [("What is this?", "A red car")],
    "What color is it?"
)
assert result_2 == "Question: What is this? Answer: A red car. Question: What color is it? Answer:", f"Got: {result_2!r}"

result_3 = format_blip2_prompt(
    [("What is this?", "A red car"), ("What color is it?", "Red")],
    "How fast can it go?"
)
expected_3 = "Question: What is this? Answer: A red car. Question: What color is it? Answer: Red. Question: How fast can it go? Answer:"
assert result_3 == expected_3, f"Got: {result_3!r}"

print("5.4 passed — all 3 test cases correct")
print("Example output:")
print(result_2)